# 🫀 퀘스트 46 · Q7-D — **`#865` 반전 감사 + 기준(reference) 선택 실험**

| | **MedKOS / `notebooks/quest46_q7d_inversion.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` |
| 앞선 실험 | `ailab-2026-0053`(Q7-B′) · `ailab-2026-0052`(Q7-B) |
| 규약 | **R4 · R10 · R11-c · R12 · R15 · R16** |
| 학습 | **0회** — 예측 캐시 + `svdb_data5.npz` 만 읽는다 |

## 이 실험이 답해야 하는 것

Q7-B′ 에서 **`#865` 의 AUROC 가 0.0595** 로 나왔다. 못 맞히는 게 아니라 **거꾸로**
맞힌다 — 점수를 뒤집으면 **0.9405** 다. 그리고 그 레코드는 비트의 **57.6% 가 S** 다.

> **S 는 「이 환자의 평소보다 이르다」는 상대량이다. 그런데 S 가 다수면 '평소' 가 곧
> S 다. 기준이 따라가 버리면 부호가 뒤집힌다.**

이게 맞다면 이 프로젝트 논지의 **가장 강한 증거**다. 그런데 그 전에 **사고 가능성을
먼저 배제**해야 하고(D-A), 배제된 뒤에는 **"RR 말고 형태로는 되나"** 를 실측해야 한다(D-C).

## 사전등록

| 관문 | 내용 | 문턱 |
|---|---|---|
| **D1** | `#865` 원본 주석(`wfdb.rdann`)의 S 개수가 캐시와 일치 → **정렬 사고 배제** | ±2 |
| **D2** | 반전이 **정보 부재가 아니다** — `1 − AUROC` | **≥ 0.85** |
| **D3** | **기준을 바꾸면 회복되나** — N군 중앙 기준(오라클)의 최고 AUROC | **≥ 0.80** |
| **D4** | **다수 의존** 기준(레코드 중앙·큰 군)의 최고 AUROC | **≥ 0.80** |
| **D4b** | ★ **다수 비의존 앵커**(2군 중 **긴 RR** 쪽을 기저로) 의 최고 AUROC | **≥ 0.80** |
| **D5** | P5 기각이 **선택 이력** 탓인가 — 신규 17개 vs 기존 55개 AUROC 분포 차 | Mann-Whitney **p < 0.05** |

**D3 · D4 · D4b 가 이 실험의 핵심**이다.

- D3 지지 + D4 기각 + **D4b 지지** → **"처방은 축을 바꾸는 게 아니라 기준을 다수에서 떼는 것"**
- D3 지지 + D4 기각 + D4b 기각 → 레코드 내부만으로는 안 된다 → 교차환자 기준·리듬 맥락 필요

⚠️ **RR 위치형(`기준 − pre_rr`)은 기준을 바꿔도 AUROC 가 안 변한다** — 상수 이동은
순위를 안 바꾸고 AUROC 는 순위 기반이다. 즉 **반전은 「거리형(절댓값)」 특징에서만**
생긴다(`‖b − ref‖` 같은 `_medref` 기반 형태 특징). RR 은 **기준 불변 대조군**으로 낸다.

⚠️ **D1 이 기각되면 거기서 멈춘다.** 정렬 사고라면 Q7-B′ 의 전수 값(0.8842)도 다시
무효다. 다른 모든 분석보다 먼저 돌린다.

## 앞선 실험에서 정정할 것 (이 노트북이 실측으로 닫는다)

**① Q7-B′ 의 P5 는 무효다 — 내 구성 오류.** 관측 통계량은 **55개체**의 28/27 격차
(0.0874)인데 귀무분포는 **전수 72개체**를 28/44 로 가른 분포였다. **통계량과 귀무분포가
다른 코호트다.** p=0.0139 는 인용하지 않는다. `【D-D】` 에서 55개체 28/27 로 다시 짠다.

**② `#865` 는 정렬 불일치 건이 아니다.** Q7-B `【Q7B-M】` 이 이미 이름을 찍었다 —
불일치는 **라벨 831 → `#848`**(N 만 3비트 차, S·V 정확)이고 `#865` 는 **라벨 848** 로
검증을 통과했다. 그래도 D1 에서 원본 주석으로 다시 대조한다(독립 확인).

**③ 검정력 계획을 SD 0.1572 로 다시 깐다.** 부분집합(28개체)에서 잰 SD 0.0833 을
모집단에 투사한 게 오류였다 — **선택된 부분집합의 산포는 모집단 산포의 하한**이다.

## ⚠️ 인용 규칙 (계속 유효)

전이 낙폭 인용 금지(대역 교란 · Q7-C) · 실험22-A·§6.5 와 직접 비교 불가(WST DS1-only fit) ·
군·개체 비교는 **AUROC 로만**(R15-d) · 오라클 기준(D3)은 **라벨을 쓴다** — 성능 주장이
아니라 **상한**이다.


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

class AssetError(RuntimeError): pass
print("CELL 0 ✅")


In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0   = 20260803
GMIN    = 2          # Q7-B 승계 · 여기서 다시 고르지 않는다
FOCUS   = 865        # 반전 개체
IDX_S, IDX_V = 1, 2
NB_BOOT = 400
RPRE    = 100        # svdb_labels._RPRE — 비트 배열에서 R 위치
P_SEG   = (0, 85)    # P 파 영역(R 앞 ~236ms @360Hz)
QRS_SEG = (85, 130)
INV_THR, ORACLE_THR, UNSUP_THR = 0.85, 0.80, 0.80

CONFIG = dict(
    exp="quest46_q7d_inversion", quest="ailab-2026-0046", step="svdb-inversion-audit",
    parent_exp=["quest46_q7bp_svdb_full", "ailab-2026-0053"],
    purpose=("#865 의 AUROC 0.0595(반전)가 사고인지 발견인지 가르고, 발견이라면 "
             "RR 말고 **형태 축**으로는 되는지, 그리고 **라벨 없이 기준을 되찾을 수** 있는지 본다"),
    dataset="SVDB 전수 · Q7-B 예측 캐시 + svdb_data5.npz (학습 0회)",
    focus_record=FOCUS, gmin=GMIN,
    corrections_closed=[
        "Q7-B′ P5 무효 — 통계량은 55개체(28/27), 귀무분포는 72개체(28/44). 【D-D】에서 재구성",
        "#865 는 매핑 불일치 건이 아니다 — 불일치는 라벨831→#848. D1 에서 원본 주석 재확인",
        "검정력 계획을 SD 0.1572 로 재작성 — 부분집합 SD 0.0833 은 모집단의 하한이었다"],
    predictions={
        "D1": "#865 원본 주석 S 개수가 캐시와 ±2 일치 (정렬 사고 배제)",
        "D2": f"1 − AUROC ≥ {INV_THR} — 반전은 정보 부재가 아니라 방향 오류다",
        "D3": f"N군 중앙 기준(오라클) 최고 AUROC ≥ {ORACLE_THR} — 기준을 바꾸면 회복된다",
        "D4": f"다수 기반 무감독 앵커 최고 AUROC ≥ {UNSUP_THR} — 라벨 없이도 기준을 찾는다",
        "D5": "신규 17개 vs 기존 55개 AUROC 분포 차 Mann-Whitney p < 0.05 (P5 기각 = 선택 이력)"},
    caveat=("**D3 는 라벨을 쓴다(오라클)** — 성능 주장이 아니라 **상한**이다. D4 가 진짜 "
            "실현 가능성이다. D1 기각이면 즉시 중단한다 — 정렬 사고면 Q7-B′ 값도 무효다. "
            "전이 낙폭 인용 금지 · 군·개체 비교는 AUROC 로만(R15-d)"))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7d_inversion", CONFIG, project=PROJECT)
run.log(f"설정 ✅ 초점 #{FOCUS} · 학습 0회")


In [ ]:
# CELL 2 — 【G0】 자산 · 매핑 (불일치 레코드를 **이름으로** 남긴다 — R16-c 보강)
PROB = os.path.join(PROJECT, "data", "q7b_svdb_probs_s5.npz")
MAPJ = os.path.join(PROJECT, "data", "svdb_id_map_q7b.json")
CNTJ = os.path.join(PROJECT, "data", "svdb_ann_counts.json")
SV5  = os.path.join(MITBIH, "svdb_data5.npz")
for p_, why in ((PROB, "Q7-B 예측 캐시"), (CNTJ, "Q7-A 주석 카운트"), (SV5, "SVDB 빌드")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}")
try:
    import wfdb
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
    importlib.invalidate_caches(); import wfdb

P = dict(np.load(PROB))
CNT = {int(k): {int(kk): vv for kk, vv in v.items()} for k, v in json.load(open(CNTJ)).items()}
Y = np.asarray(P["y_cross"]); REC = np.asarray(P["rec_cross"]).astype(int)
recs = [int(r) for r in wfdb.get_record_list("svdb")]      # ⛔ fallback 없음 (R16)
if len(recs) < 2:
    raise AssetError("SVDB 목록을 못 받았다 — 연속 번호로 대체하지 않는다")

labels = sorted(int(x) for x in np.unique(REC))
if all(l in recs for l in labels):
    MAP = {l: l for l in labels}
elif os.path.exists(MAPJ):
    MAP = {int(k): int(v) for k, v in json.load(open(MAPJ)).items()}
else:
    CONT = [min(recs) + i for i in range(len(recs))]
    if labels != CONT:
        raise AssetError("라벨이 목록 밖인데 연속 가정과도 안 맞는다")
    MAP = {l: recs[l - CONT[0]] for l in labels}
EXCL17 = sorted(MAP[l] for l in labels if l not in recs)   # Q7-B 가 통째로 놓쳤던 참 레코드
REC = np.array([MAP[int(r)] for r in REC], dtype=np.int64)
assert len(np.unique(REC)) == len(labels) and not (set(REC.tolist()) - set(recs))

# ── 재검증: **불일치를 이름으로 전부 출력한다.** 개수만 세면 어느 게 틀렸는지 기록에 안 남는다.
CNT3 = {r: (v.get(0, 0), v.get(1, 0), v.get(2, 0)) for r, v in CNT.items()}
mism = []
for r in np.unique(REC):
    o = tuple(int((Y[REC == r] == k).sum()) for k in range(3)); a = CNT3.get(int(r))
    if not a or any(abs(x - y) > 2 for x, y in zip(o, a)):
        mism.append((int(r), o, a))
run.log("\n" + "=" * 100)
run.log("【G0】 자산 · 매핑")
run.log("=" * 100)
run.log(f"  예측 {len(Y):,}비트 · 레코드 {len(np.unique(REC))}개 · Q7-B 가 놓쳤던 {len(EXCL17)}개: {EXCL17}")
run.log(f"  (N,S,V) 재검증 불일치 **{len(mism)}건**")
for r, o, a in mism:
    run.log(f"    #{r}  예측 {o}  vs 주석 {a}   차 {tuple(x-y for x, y in zip(o, a))}")
if any(r == FOCUS for r, _, _ in mism):
    raise AssetError(f"#{FOCUS} 가 불일치 목록에 있다 — 정렬 사고 가능성. 여기서 멈춘다")
run.log(f"  ✅ #{FOCUS} 는 불일치 목록에 **없다**")
CONFIG["mismatch"] = [{"rec": r, "obs": list(o), "ann": list(a) if a else None} for r, o, a in mism]
CONFIG["excluded_in_q7b"] = EXCL17
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【D-A】 `#865` 감사 — **다른 모든 것보다 먼저.** 기각이면 중단
from sklearn.metrics import roc_auc_score
run.log("\n" + "=" * 100)
run.log(f"【D-A】 #{FOCUS} 감사 — 사고(정렬)인가 발견(맥락 반전)인가")
run.log("=" * 100)

m = np.where(REC == FOCUS)[0]
t = (Y[m] == IDX_S)
sc = P["v2_cross_raw"].mean(0)[m, IDX_S]
auc = float(roc_auc_score(t.astype(int), sc))
prev = float(t.mean())
run.log(f"  캐시 — 비트 {len(m):,} · S {int(t.sum()):,} · 유병률 {prev:.4f}"
        f" · AUROC {auc:.4f} · **반전 {1-auc:.4f}**")

# ── D1: 원본 주석과 직접 대조 (독립 확인 — 캐시를 안 거친다)
ann = wfdb.rdann(str(FOCUS), "atr", pn_dir="svdb")
_A = {'N': 0, 'L': 0, 'R': 0, 'e': 0, 'j': 0, 'A': 1, 'a': 1, 'J': 1, 'S': 1, 'V': 2, 'E': 2}
raw = {0: 0, 1: 0, 2: 0}
for s_ in ann.symbol:
    k = _A.get(s_)
    if k is not None: raw[k] += 1
run.log(f"\n  원본 주석 직독 — N {raw[0]:,} · S {raw[1]:,} · V {raw[2]:,}"
        f"  (유병률 {raw[1]/max(sum(raw.values()),1):.4f})")
d1_ok = abs(raw[1] - int(t.sum())) <= 2
VERD = {}
def g_(k, ok, d):
    VERD[k] = "✅ 지지" if ok else "❌ 기각"; run.log(f"  {k:<4}{VERD[k]}  {d}")
g_("D1", d1_ok, f"원본 S {raw[1]:,} vs 캐시 S {int(t.sum()):,} (차 {raw[1]-int(t.sum())})")
if not d1_ok:
    raise AssetError(f"D1 기각 — #{FOCUS} 의 주석과 캐시가 안 맞는다. 정렬 사고다. "
                     "Q7-B′ 전수 값도 다시 무효다. 여기서 멈춘다")

# ── 리듬 라벨(aux_note) — 무엇인지 **데이터로** 말하게 한다
aux = [str(a).strip("\x00").strip() for a in (getattr(ann, "aux_note", []) or []) if str(a).strip("\x00").strip()]
from collections import Counter
run.log(f"\n  리듬 주석(aux_note) {len(aux)}건 · 종류 {Counter(aux).most_common(6)}")

# ── D2: 반전이 정보 부재가 아님
rng = np.random.RandomState(SEED0)
bs = []
for _ in range(NB_BOOT):
    j = rng.randint(0, len(m), len(m)); tj = t[j]
    if 0 < tj.sum() < len(tj): bs.append(roc_auc_score(tj.astype(int), sc[j]))
se = float(np.std(bs, ddof=1))
g_("D2", (1 - auc) >= INV_THR,
   f"반전 AUROC **{1-auc:.4f}** ≥ {INV_THR} (SE {se:.4f}) — 표현은 S/N 을 가른다. **방향만** 틀렸다")

# ── RR 구조: S 가 규칙적이고 N 이 이탈하는가
d5 = np.load(SV5, allow_pickle=True)
keep = d5["y3"] >= 0
assert int(keep.sum()) == len(Y), f"svdb_data5 와 예측 캐시 길이 불일치 {int(keep.sum())} vs {len(Y)}"
PRE = d5["pre_rr"][keep].astype(float)[m]
rs, rn = PRE[t], PRE[~t]
run.log(f"\n  RR 구조 (샘플 @360Hz) — S: 중앙 {np.median(rs):.0f} · CV {np.std(rs)/max(np.mean(rs),1e-9):.3f}"
        f"  |  N: 중앙 {np.median(rn):.0f} · CV {np.std(rn)/max(np.mean(rn),1e-9):.3f}")
run.log(f"  레코드 전체 중앙 RR {np.median(PRE):.0f} — S 가 다수라 **중앙값이 S 쪽에 있다**"
        if np.median(PRE) < (np.median(rs) + np.median(rn)) / 2 else
        f"  레코드 전체 중앙 RR {np.median(PRE):.0f}")
CONFIG["focus"] = dict(rec=FOCUS, n=len(m), s=int(t.sum()), prev=prev, auroc=auc,
                       inverted=1 - auc, se=se, ann_raw=raw, aux=Counter(aux).most_common(8),
                       rr_med_s=float(np.median(rs)), rr_med_n=float(np.median(rn)),
                       rr_med_all=float(np.median(PRE)))
run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — 【D-B】 반전 전수 조사 · 유병률 상관 · rho 감도
def per_rec(score, idx, recs_, nboot=0):
    out = {}
    rng2 = np.random.RandomState(SEED0 + 1)
    for r in recs_:
        mm = np.where(REC == r)[0]
        if len(mm) < 3: continue
        tt = (Y[mm] == idx)
        if not (tt.any() and not tt.all()) or tt.sum() < GMIN: continue
        a_ = float(roc_auc_score(tt.astype(int), score[mm]))
        s_ = np.nan
        if nboot:
            v = []
            for _ in range(nboot):
                j = rng2.randint(0, len(mm), len(mm)); tj = tt[j]
                if 0 < tj.sum() < len(tj): v.append(roc_auc_score(tj.astype(int), score[mm][j]))
            s_ = float(np.std(v, ddof=1)) if len(v) > 2 else np.nan
        out[int(r)] = dict(auroc=a_, se=s_, pos=int(tt.sum()), prev=float(tt.mean()), n=len(mm))
    return out

SC_S = P["v2_cross_raw"].mean(0)[:, IDX_S]
SC_V = P["v2_cross_raw"].mean(0)[:, IDX_V]
ALL = [int(r) for r in np.unique(REC)]
PS = per_rec(SC_S, IDX_S, ALL, nboot=NB_BOOT)
PV = per_rec(SC_V, IDX_V, ALL)
run.log("\n" + "=" * 100)
run.log("【D-B】 반전 전수 조사")
run.log("=" * 100)
au = np.array([PS[r]["auroc"] for r in sorted(PS)])
pv = np.array([PS[r]["prev"] for r in sorted(PS)])
po = np.array([PS[r]["pos"] for r in sorted(PS)])
rr = np.array(sorted(PS))
inv = rr[au < 0.5]
run.log(f"  채점 개체 {len(rr)} · **AUROC < 0.5 인 개체 {len(inv)}개**: "
        + ", ".join(f"#{r}({PS[r]['auroc']:.3f} · 유병률 {PS[r]['prev']:.3f})" for r in inv))
run.log(f"  유병률 > 0.5 인 개체: "
        + (", ".join(f"#{r}({PS[r]['prev']:.3f} · AUROC {PS[r]['auroc']:.3f})"
                     for r in rr[pv > 0.5]) or "없음"))
rho_p, p_p = stats.spearmanr(pv, au)
run.log(f"\n  유병률 vs AUROC 스피어만 rho={rho_p:+.3f} p={p_p:.4f}")
k = rr != FOCUS
rho_p2, p_p2 = stats.spearmanr(pv[k], au[k])
run.log(f"    #{FOCUS} 제외 → rho={rho_p2:+.3f} p={p_p2:.4f}"
        f"   {'← 계통이 살아남는다' if p_p2 < 0.05 else '← **극단 1례 이야기였다**'}")
rho_n, p_n = stats.spearmanr(po, au); rho_n2, p_n2 = stats.spearmanr(po[k], au[k])
run.log(f"  양성수 vs AUROC rho={rho_n:+.3f} p={p_n:.4f}"
        f"  ·  #{FOCUS} 제외 rho={rho_n2:+.3f} p={p_n2:.4f}"
        f"   {'← 살아남는다' if p_n2 < 0.05 else '← **#865 에 의존했다**'}")

# ── 매크로 옆에 **분포 요약**을 상시 병기한다 (한 숫자로 대표되지 않는다)
q = np.percentile(au, [0, 25, 50, 75, 100])
run.log(f"\n  개체별 AUROC 분포 — 최소 {q[0]:.4f} · Q1 {q[1]:.4f} · **중앙 {q[2]:.4f}**"
        f" · Q3 {q[3]:.4f} · 최대 {q[4]:.4f} · 매크로 {au.mean():.4f} · SD {au.std(ddof=1):.4f}")
run.log(f"    #{FOCUS} 제외 → 매크로 {au[k].mean():.4f} · SD {au[k].std(ddof=1):.4f}")

# ── 검정력 계획 (R15 · SD 를 **모집단**에서 잰 값으로)
run.log(f"\n  ── 검정력 계획 (폭 = 2·1.96·SD/√n) ──")
for tag, sd in (("전수 SD", au.std(ddof=1)), (f"#{FOCUS} 제외 SD", au[k].std(ddof=1))):
    row = " · ".join(f"폭{w:.2f}→n{int(np.ceil((2*1.96*sd/w)**2)):>4}" for w in (0.08, 0.06, 0.05, 0.04))
    run.log(f"    {tag} {sd:.4f}: {row}  |  SVDB 78 전량 → 폭 {2*1.96*sd/np.sqrt(78):.4f}")
run.log("    ⚠️ 부분집합에서 잰 SD 를 모집단에 투사하지 않는다 — **선택된 부분집합의 산포는 하한**이다")
CONFIG["sweep"] = {"per_record": {str(r): PS[r] for r in sorted(PS)},
                   "inverted": [int(x) for x in inv],
                   "rho_prev": [float(rho_p), float(p_p)], "rho_prev_ex": [float(rho_p2), float(p_p2)],
                   "rho_pos": [float(rho_n), float(p_n)], "rho_pos_ex": [float(rho_n2), float(p_n2)],
                   "auroc_quartiles": q.tolist(), "sd": float(au.std(ddof=1)),
                   "sd_ex_focus": float(au[k].std(ddof=1))}
run.save_json("config", CONFIG)


In [ ]:
# CELL 5 — 【D-C】 ★ **기준 선택 실험** — "RR 말고 형태로는 되나"
#
#  ★★ 픽스처가 개념 오류를 하나 잡았다. 처음엔 RR 축에도 기준 셋을 다 돌렸는데,
#     **위치형 점수는 기준을 바꿔도 AUROC 가 안 변한다**: `base − pre` 는 base 가 바뀌면
#     전부 같은 상수만큼 이동하고, AUROC 는 **순위 기반**이라 상수 이동에 불변이다.
#     → **반전은 「거리형(절댓값) 특징」에서만 생긴다.** `‖b − ref‖` 는 ref 가 바뀌면
#       순위 자체가 바뀐다. `_medref` 기반 형태 특징이 정확히 이 형태다.
#     그래서 RR 은 **기준 불변 대조군**으로 한 번만 내고, 격자는 형태 축에 건다.
#
#  ★★ 그리고 방향을 절대 접지 않는다. `max(AUROC, 1−AUROC)` 로 '분리도' 만 보면
#     **완벽히 뒤집힌 기준도 만점**을 받는다 — 방향을 모르면 못 쓴다는 게 요점인데
#     그 요점이 지워진다. **방향 있는 AUROC 가 주 지표**, `max(·)` 는 '정보량' 으로 병기.
#
#  기준 후보 넷:
#    · 다수결(레코드 중앙 = 현행 `_medref`)  ← 다수가 S 면 끌려간다
#    · 오라클(N 군 중앙)                      ← **라벨 사용**. 성능이 아니라 **상한**
#    · 무감독·큰 군                            ← 다수결의 클러스터판. 같은 함정
#    · 무감독·**긴 RR 군**                     ← ★ **다수에 의존하지 않는 앵커**.
#      「이소성 박동은 이르다」는 생리 사전지식만 쓴다. 개수를 안 본다.
BEAT = d5["beat"][keep]
POST = d5["post_rr"][keep].astype(float)
run.log("\n" + "=" * 100)
run.log(f"【D-C】 기준 선택 실험 — #{FOCUS} (유병률 {prev:.3f})")
run.log("=" * 100)

def seg_dist(B, ref, seg):
    s0, s1 = seg
    d = B[:, :, s0:s1] - ref[None, :, s0:s1]
    return np.sqrt((d ** 2).sum(axis=(1, 2)))

Bm, tm, pre_m = BEAT[m], t, PRE

# ── 무감독 2군 분할 (P 영역 k-means) — 라벨 미사용
from sklearn.cluster import KMeans
_feat = Bm[:, :, P_SEG[0]:P_SEG[1]].reshape(len(Bm), -1)
_lab = KMeans(2, n_init=10, random_state=SEED0).fit_predict(_feat)
c0, c1 = (_lab == 0), (_lab == 1)
big = 0 if c0.sum() >= c1.sum() else 1                       # 다수 앵커
longrr = 0 if np.median(pre_m[c0]) >= np.median(pre_m[c1]) else 1   # ★ 긴 RR 앵커
run.log(f"  무감독 2군 — 군0 {int(c0.sum()):,}비트(S비율 {tm[c0].mean():.3f} · 중앙RR {np.median(pre_m[c0]):.0f})"
        f" | 군1 {int(c1.sum()):,}비트(S비율 {tm[c1].mean():.3f} · 중앙RR {np.median(pre_m[c1]):.0f})")
run.log(f"    다수 앵커 → 군{big} (S비율 {tm[_lab==big].mean():.3f})"
        + ("  ⛔ **S 를 '평소' 로 삼는다**" if tm[_lab == big].mean() > 0.5 else ""))
run.log(f"    긴RR 앵커 → 군{longrr} (S비율 {tm[_lab==longrr].mean():.3f})"
        + ("  ✅ 개수를 안 보고 기저를 골랐다" if tm[_lab == longrr].mean() <= 0.5 else "  ⛔ 이것도 S 를 골랐다"))

REFS = [("다수결(_medref)", np.median(Bm, axis=0), "maj"),
        ("오라클(N군 중앙)", np.median(Bm[~tm], axis=0), "oracle"),
        ("무감독·큰 군", np.median(Bm[_lab == big], axis=0), "maj"),
        ("무감독·긴RR 군", np.median(Bm[_lab == longrr], axis=0), "anchor")]

ROWS = []
# ★ '뒤집힘' 은 0.5 미만이라는 것만으로 부족하다. 신호가 아예 없는 축(예: S/N 의 QRS 가
#   같으면 QRS 거리는 0.5 근처를 무작위로 오간다)에서는 0.48 도 '뒤집힘' 으로 찍힌다.
#   → **신호가 있는 축(정보량 ≥ INFO_MIN)에서만 '뒤집힘' 을 의미 있게 본다.**
INFO_MIN = 0.60
def add(axis, refname, kind, sig):
    a_ = float(roc_auc_score(tm.astype(int), sig))           # ★ 방향 있는 AUROC
    info = float(max(a_, 1 - a_))
    ROWS.append(dict(axis=axis, ref=refname, kind=kind, auroc=a_, info=info,
                     signal=bool(info >= INFO_MIN),
                     inverted=bool(a_ < 0.5 and info >= INFO_MIN)))

# ── 대조군: RR 위치형 (기준 불변 — 한 번만)
a_rr = float(roc_auc_score(tm.astype(int), np.median(pre_m) - pre_m))
run.log(f"\n  [대조] RR 위치형 「기준보다 이르다」 AUROC **{a_rr:.4f}**")
run.log("     ⚠️ 이 값은 **기준을 뭘로 잡든 같다** — 상수 이동은 순위를 안 바꾼다.")
run.log("        즉 **RR 위치형은 구조적으로 반전될 수 없다.** 반전은 거리형에서만 생긴다.")
ROWS.append(dict(axis="RR·위치형", ref="(기준 불변)", kind="invariant", auroc=a_rr,
                 info=float(max(a_rr, 1 - a_rr)), signal=bool(max(a_rr, 1-a_rr) >= 0.60),
                 inverted=bool(a_rr < 0.5 and max(a_rr, 1 - a_rr) >= 0.60)))

# ── 형태 거리형 × 기준 4종
for segnm, seg in (("전파형", (0, 300)), ("P영역", P_SEG), ("QRS영역", QRS_SEG)):
    for nm_, ref_, kind in REFS:
        add(f"형태·{segnm}", nm_, kind, seg_dist(Bm, ref_, seg))

# ── 음성 대조: **라벨을 섞는다**(기준을 섞는 게 아니다).
#    기준을 무작위 부분집합에서 만들어도 그 중앙은 여전히 다수 모드에 앉으므로 null 이 아니다.
rngc = np.random.RandomState(SEED0 + 5)
dP = seg_dist(Bm, np.median(Bm, axis=0), P_SEG)
ctrl = [roc_auc_score(rngc.permutation(tm).astype(int), dP) for _ in range(50)]
nc_ok = abs(np.median(ctrl) - 0.5) < 0.05
run.log(f"\n  음성 대조(라벨 셔플 50회) AUROC 중앙 {np.median(ctrl):.4f}"
        f" [{np.min(ctrl):.4f}, {np.max(ctrl):.4f}]  {'✅' if nc_ok else '⛔ 지표가 샌다'}")

run.log(f"\n  {'축':<14}{'기준':<20}{'AUROC(방향O)':>13}{'정보량':>9}{'뒤집힘':>8}   비고")
KIND = {"invariant": "기준 불변 대조", "maj": "다수 의존", "oracle": "라벨 사용(상한)", "anchor": "다수 비의존 앵커"}
for r_ in ROWS:
    flag = "  예" if r_["inverted"] else ("  —" if r_["signal"] else " 신호없음")
    run.log(f"  {r_['axis']:<14}{r_['ref']:<20}{r_['auroc']:>13.4f}{r_['info']:>9.4f}"
            f"{flag:>8}   {KIND[r_['kind']]}")

best = lambda k: max((r_ for r_ in ROWS if r_["kind"] == k), key=lambda x: x["auroc"], default=None)
bo, bm_, ba = best("oracle"), best("maj"), best("anchor")
run.log("")
g_("D3", bo and bo["auroc"] >= ORACLE_THR,
   f"오라클 최고 **{bo['auroc']:.4f}** ({bo['axis']}) ≥ {ORACLE_THR} — 기준만 바꾸면 회복되나")
g_("D4", bm_ and bm_["auroc"] >= UNSUP_THR,
   f"**다수 의존** 기준 최고 **{bm_['auroc']:.4f}** ({bm_['axis']} · {bm_['ref']}) ≥ {UNSUP_THR}")
g_("D4b", ba and ba["auroc"] >= UNSUP_THR,
   f"**다수 비의존 앵커(긴RR 군)** 최고 **{ba['auroc']:.4f}** ({ba['axis']}) ≥ {UNSUP_THR}"
   f"  ← 처방 후보")

inv_maj = sorted({r_["axis"] for r_ in ROWS if r_["inverted"] and r_["kind"] == "maj"})
run.log(f"\n  다수 의존 기준에서 뒤집힌 축: {inv_maj or '없음'}")
if VERD["D4"].startswith("❌") and VERD["D4b"].startswith("✅"):
    run.log("\n  ★ **결론이 선명하다**: 정보는 있다(D3). 다수결 기준으로는 뒤집힌다(D4).")
    run.log("     그런데 **개수를 안 보고 「이소성은 이르다」만 쓰는 앵커로는 회복된다**(D4b).")
    run.log("     → 처방은 '형태 축으로 옮기기' 가 아니라 **'기준을 다수에서 떼기'** 다.")
elif VERD["D4"].startswith("❌") and VERD["D4b"].startswith("❌"):
    run.log("\n  ⛔ 다수 앵커도 긴RR 앵커도 실패했다. 레코드 내부 정보만으로는 기준을 못 잡는다.")
    run.log("     → **교차환자 기준**(다른 환자들의 N 분포)이나 **리듬 수준 맥락**(aux_note)이 필요하다.")
CONFIG["reference_grid"] = ROWS
CONFIG["rr_positional_auroc"] = a_rr
CONFIG["neg_control"] = dict(median=float(np.median(ctrl)), lo=float(np.min(ctrl)),
                             hi=float(np.max(ctrl)), ok=bool(nc_ok))
CONFIG["unsup_cluster"] = dict(
    big_s_frac=float(tm[_lab == big].mean()), longrr_s_frac=float(tm[_lab == longrr].mean()),
    big_is_longrr=bool(big == longrr),
    n0=int(c0.sum()), n1=int(c1.sum()),
    rr0=float(np.median(pre_m[c0])), rr1=float(np.median(pre_m[c1])))
CONFIG["inverted_axes_majority"] = inv_maj
run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【D-D】 선택 이력 — Q7-B′ P5 를 **올바르게** 다시 짠다
#
#  ★ Q7-B′ 의 P5 는 무효였다(내 구성 오류): 관측 통계량은 **55개체**의 28/27 격차인데
#    귀무분포는 **72개체**를 28/44 로 가른 것이었다. 통계량과 귀무분포가 다른 코호트다.
run.log("\n" + "=" * 100)
run.log("【D-D】 선택 이력 — P5 재구성 + 신규 17 vs 기존 55")
run.log("=" * 100)
SC_ANN = {r: c.get(1, 0) for r, c in CNT.items()}
order = sorted(SC_ANN, key=lambda r: (-SC_ANN[r], r))
TEST_ANN, DEV_ANN = set(order[0::2]), set(order[1::2])
inv_map = {v: k for k, v in MAP.items()}                 # 참 → Q7-B 라벨
scored = sorted(PS)
OLD = [r for r in scored if r not in EXCL17]
NEW = [r for r in scored if r in EXCL17]
TEST55 = [r for r in OLD if inv_map[r] in TEST_ANN]
DEV55  = [r for r in OLD if inv_map[r] in DEV_ANN]
run.log(f"  기존(Q7-B 가 본) {len(OLD)}개 = TEST {len(TEST55)} + DEV {len(DEV55)}"
        f"  ·  신규(Q7-B 가 놓친) {len(NEW)}개")

aT = np.array([PS[r]["auroc"] for r in TEST55]); aD = np.array([PS[r]["auroc"] for r in DEV55])
aO = np.array([PS[r]["auroc"] for r in OLD]);    aN = np.array([PS[r]["auroc"] for r in NEW])
run.log(f"  TEST 매크로 {aT.mean():.4f} · DEV 매크로 {aD.mean():.4f} · 격차 {abs(aT.mean()-aD.mean()):.4f}")
run.log(f"  기존 매크로 {aO.mean():.4f} (SD {aO.std(ddof=1):.4f}) · "
        f"신규 매크로 {aN.mean():.4f} (SD {aN.std(ddof=1):.4f})")

# ── P5 재구성: **그 55개체 안에서만** 28/27 순열
obs = abs(aT.mean() - aD.mean())
rng3 = np.random.RandomState(SEED0 + 13)
n1 = len(TEST55); NPERM = 20000
g55 = np.empty(NPERM)
for i in range(NPERM):
    pm = rng3.permutation(len(aO))
    g55[i] = abs(aO[pm[:n1]].mean() - aO[pm[n1:]].mean())
p55 = float((g55 >= obs).mean())
run.log(f"\n  P5 재구성 — 55개체 안에서 {n1}/{len(aO)-n1} 순열: 관측 {obs:.4f}"
        f" · 중앙 {np.median(g55):.4f} · 95분위 {np.percentile(g55,95):.4f} → **p = {p55:.4f}**")
run.log(f"    (무효였던 Q7-B′ 값: 72개체 28/44 순열에서 p = 0.0139)")

# ── D5: 신규 17 이 기존 55 와 계통적으로 다른가
U, pU = stats.mannwhitneyu(aN, aO, alternative="two-sided")
g_("D5", pU < 0.05,
   f"신규 {len(aN)} vs 기존 {len(aO)} Mann-Whitney **p = {pU:.4f}**"
   f" (중앙 {np.median(aN):.4f} vs {np.median(aO):.4f})")
if pU < 0.05:
    run.log("    → P5 기각은 **SVDB 의 성질이 아니라 오염된 선택 필터의 흔적**이다.")
else:
    run.log("    → 신규군이 계통적으로 다르지 않다. 격차의 원인을 다른 데서 찾아야 한다"
            f" (유력 후보: #{FOCUS} 가 DEV 에 있었다).")
run.log(f"  참고 — #{FOCUS} 는 {'TEST' if FOCUS in TEST55 else ('DEV' if FOCUS in DEV55 else '신규')} 에 있다")
CONFIG["selection_history"] = dict(
    n_old=len(OLD), n_new=len(NEW), n_test=len(TEST55), n_dev=len(DEV55),
    macro_test=float(aT.mean()), macro_dev=float(aD.mean()),
    macro_old=float(aO.mean()), macro_new=float(aN.mean()),
    p5_correct=p55, p5_obs=float(obs), mw_p=float(pU),
    focus_side=("TEST" if FOCUS in TEST55 else ("DEV" if FOCUS in DEV55 else "NEW")))
run.save_json("config", CONFIG)


In [ ]:
# CELL 7 — 【D-E】 짝지은 S/V 비교 (교집합 코호트) — 이질성을 상쇄해 검정력을 번다
run.log("\n" + "=" * 100)
run.log("【D-E】 S vs V 짝지은 비교")
run.log("=" * 100)
both = sorted(set(PS) & set(PV))
run.log(f"  S 채점 {len(PS)} · V 채점 {len(PV)} · **둘 다 되는 교집합 {len(both)}**")
dS = np.array([PS[r]["auroc"] for r in both]); dV = np.array([PV[r]["auroc"] for r in both])
diff = dS - dV
rng4 = np.random.RandomState(SEED0 + 21)
bd = [float(diff[rng4.randint(0, len(diff), len(diff))].mean()) for _ in range(4000)]
lo_, hi_ = np.percentile(bd, [2.5, 97.5])
run.log(f"  교집합에서 — S 매크로 {dS.mean():.4f} · V 매크로 {dV.mean():.4f}")
run.log(f"  **짝지은 차 (S − V) {diff.mean():+.4f}** [{lo_:+.4f}, {hi_:+.4f}] 폭 {hi_-lo_:.4f}")
run.log(f"    비짝지음 격차(전수): {0.8842-0.9426:+.4f} — CI 가 겹쳤다")
run.log(f"    S 가 더 나쁜 개체 {int((diff<0).sum())}/{len(diff)} · 더 좋은 개체 {int((diff>0).sum())}")
run.log("  → 같은 환자에서 뺐으므로 개체 간 이질성(SD 0.157)이 상쇄된다."
        f" 폭 {hi_-lo_:.4f} 가 비짝지음보다 좁으면 검정력을 번 것이다.")
kk = np.array(both) != FOCUS
run.log(f"    #{FOCUS} 제외 → 짝지은 차 {diff[kk].mean():+.4f}")
CONFIG["paired_sv"] = dict(n=len(both), macro_s=float(dS.mean()), macro_v=float(dV.mean()),
                           diff=float(diff.mean()), lo=float(lo_), hi=float(hi_),
                           diff_ex_focus=float(diff[kk].mean()),
                           n_worse=int((diff < 0).sum()), n_better=int((diff > 0).sum()))
run.log("\n  " + "  ".join(f"{k}: {v}" for k, v in VERD.items()))
CONFIG["result"] = {"verdicts": VERD}
run.save_json("config", CONFIG)


In [ ]:
# CELL 8 — 그림 + 마무리
import matplotlib.pyplot as plt
S = CONFIG["sweep"]; rr_ = np.array([int(k) for k in S["per_record"]])
au_ = np.array([S["per_record"][k]["auroc"] for k in S["per_record"]])
pv_ = np.array([S["per_record"][k]["prev"] for k in S["per_record"]])
fig, ax = plt.subplots(1, 3, figsize=(19, 4.6))
ax[0].scatter(pv_, au_, s=28); ax[0].axhline(.5, color="crimson", ls="--", label="AUROC 0.5")
for i in np.where(au_ < 0.5)[0]:
    ax[0].annotate(f"#{rr_[i]}", (pv_[i], au_[i]), fontsize=9, xytext=(4, 4), textcoords="offset points")
ax[0].set_xlabel("S 유병률"); ax[0].set_ylabel("AUROC")
ax[0].set_title("① 유병률 vs AUROC (전수)"); ax[0].legend(); ax[0].grid(alpha=.3)

G = CONFIG["reference_grid"]
lbl = [f"{g['axis']}\n{g['ref'].split('(')[0]}" for g in G]
col = ["C2" if g["oracle"] else ("C3" if g["auroc"] < UNSUP_THR else "C0") for g in G]
ax[1].bar(range(len(G)), [g["auroc"] for g in G], color=col)
ax[1].axhline(ORACLE_THR, color="gray", ls=":", label=f"문턱 {ORACLE_THR}")
ax[1].set_xticks(range(len(G))); ax[1].set_xticklabels(lbl, rotation=60, ha="right", fontsize=7)
ax[1].set_ylabel("분리도(AUROC)"); ax[1].set_title(f"② #{FOCUS} 기준 선택 격자 (초록=오라클)")
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3, axis="y")

H = CONFIG["selection_history"]
ax[2].hist(g55, bins=50, alpha=.7)
ax[2].axvline(H["p5_obs"], color="crimson", lw=2, label=f"관측 {H['p5_obs']:.4f} (p={H['p5_correct']:.3f})")
ax[2].set_xlabel("|반쪽 매크로 차|"); ax[2].set_title("③ P5 재구성 — 55개체 28/27 순열")
ax[2].legend(fontsize=8); ax[2].grid(alpha=.3)
plt.tight_layout(); run.save_fig("q7d_inversion", fig); plt.show()

run.finish(CONFIG.get("result", {"verdicts": VERD}))
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step svdb-inversion-audit`")
